# Step 5 — Register in RHOAI Model Registry and deploy with KServe

Three things happen here:
1. Upload the `best.onnx` weights (exported in notebook 04) to S3.
2. Register a new `ModelVersion` in the RHOAI Model Registry pointing at the S3 URI.
3. Create a KServe `InferenceService` served by **OpenVINO Model Server (OVMS)** — the same runtime the lab's Fraud Detection demo uses. OVMS loads ONNX natively and speaks KServe v2.

**Cluster prerequisites** (the workshop lab environment provides these):
- An S3-compatible object store reachable from the workbench (ODF / MinIO / RGW).
- The RHOAI Model Registry component enabled (`kubectl get modelregistry -A`).
- The RHOAI ServingRuntime template `kserve-ovms` available (it ships with RHOAI by default). Notebook 05 instantiates it into this project if it isn't already present.

In [1]:
%%capture
!pip install boto3==1.35.55 botocore==1.35.55 model-registry kubernetes requests pyyaml

In [2]:
import os
import time
from pathlib import Path

import boto3
import botocore

## 5a. Upload `best.onnx` to S3 in OVMS layout

OpenVINO Model Server discovers model versions by walking numbered subdirectories under `model_path`. So the object layout must be:

```
s3://$BUCKET/models/ppe-yolov8n-vlm/
  └── 1/                 # the MODEL_VERSION — numeric subdirectory
      └── model.onnx
      └── data.yaml      # class names, for humans / audit
```

KServe's `storageUri` will point at the parent prefix; OVMS finds `1/model.onnx` inside. Uses the standard RHOAI data-connection env vars injected into the workbench.

In [ ]:
import shutil

aws_access_key_id     = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
endpoint_url          = os.environ.get("AWS_S3_ENDPOINT")
region_name           = os.environ.get("AWS_DEFAULT_REGION")
bucket_name           = os.environ.get("AWS_S3_BUCKET")

# Fail loudly if the workbench's Data Connection isn't attached — same guard as
# the upstream Fraud Detection tutorial (rh-aiservices-bu/fraud-detection).
if not all([aws_access_key_id, aws_secret_access_key, endpoint_url, region_name, bucket_name]):
    raise ValueError(
        "One or more S3 connection variables are empty. "
        "Attach a Data Connection to this workbench (AWS_ACCESS_KEY_ID / "
        "AWS_SECRET_ACCESS_KEY / AWS_S3_ENDPOINT / AWS_DEFAULT_REGION / AWS_S3_BUCKET)."
    )

MODEL_NAME    = "ppe-yolov8n-vlm"
MODEL_VERSION = "1"                        # must be a numeric string — OVMS treats it as a version dir
S3_PREFIX     = f"models/{MODEL_NAME}"     # parent; version subdir ({MODEL_VERSION}/) sits underneath

# Find the newest ONNX export from notebook 04.
candidates = sorted(Path("runs").rglob("best.onnx"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.onnx found under runs/ — re-run notebook 04's export cell first."
LOCAL_ONNX = candidates[0]
print(f"Using ONNX weights: {LOCAL_ONNX}  ({LOCAL_ONNX.stat().st_size / 1e6:.1f} MB)")

# Stage the model into the OVMS-required layout under a local `models/` tree,
# then mirror-upload that tree. Matches the Fraud tutorial exactly.
local_models_dir = Path("models") / MODEL_NAME / MODEL_VERSION
local_models_dir.mkdir(parents=True, exist_ok=True)
shutil.copyfile(LOCAL_ONNX, local_models_dir / "model.onnx")
shutil.copyfile("data-vlm.yaml", local_models_dir / "data.yaml")
print(f"Staged at: {local_models_dir}/")

session = boto3.session.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
s3_resource = session.resource(
    "s3",
    config=botocore.client.Config(signature_version="s3v4"),
    endpoint_url=endpoint_url,
    region_name=region_name,
)
bucket = s3_resource.Bucket(bucket_name)


def upload_directory_to_s3(local_directory, s3_prefix):
    num_files = 0
    for root, _, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            print(f"  {file_path} -> s3://{bucket_name}/{s3_key}")
            bucket.upload_file(file_path, s3_key)
            num_files += 1
    return num_files


def list_objects(prefix):
    for obj in bucket.objects.filter(Prefix=prefix):
        print(f"  {obj.key}  ({obj.size / 1e6:.2f} MB)")

In [ ]:
print(f"Bucket contents BEFORE (prefix={S3_PREFIX!r}):")
list_objects(S3_PREFIX)

print("\nUploading…")
num_files = upload_directory_to_s3("models", "models")
if num_files == 0:
    raise ValueError(
        "No files uploaded. Did the staging step fail? "
        "Expected models/ppe-yolov8n-vlm/1/model.onnx to exist locally."
    )

print(f"\nBucket contents AFTER (prefix={S3_PREFIX!r}):")
list_objects(S3_PREFIX)

# storageUri points at the PARENT directory. OVMS walks into 1/ (and 2/, 3/ if they appear later).
s3_uri = f"s3://{bucket_name}/{S3_PREFIX}/"
print(f"\nKServe storageUri: {s3_uri}")

## 5b. Register the model in RHOAI Model Registry

`MODEL_REGISTRY_URL` is the route of the Model Registry service on your cluster — the lab will inject it as an env var. Get it manually with:

```bash
oc get route -n rhoai-model-registries modelregistry-sample -o jsonpath='{.spec.host}'
```

In [ ]:
import csv
import datetime

import yaml
from model_registry import ModelRegistry

# --- Harvest training hyperparameters and final metrics from the Ultralytics run ---
# Both files sit beside best.onnx under weights/. `LOCAL_ONNX` was set in 5a.
run_dir = LOCAL_ONNX.parent.parent  # weights/best.onnx -> weights/ -> run root

args_path    = run_dir / "args.yaml"
results_path = run_dir / "results.csv"

train_args = yaml.safe_load(args_path.read_text()) if args_path.is_file() else {}
final_row  = {}
if results_path.is_file():
    rows = list(csv.DictReader(results_path.open()))
    if rows:
        final_row = rows[-1]

def _fmt(v, digits=4):
    try:
        return f"{float(v):.{digits}f}"
    except (TypeError, ValueError):
        return "" if v is None else str(v)

# --- Build the ModelVersion metadata (this-run-specific) ---
# Stored as `custom_properties` on the ModelVersion. RHOAI dashboard convention
# (from ODH source): empty-string values render as "Labels"; non-empty values
# (strings, ints, floats) render as "Properties".
metadata = {
    # Labels (empty string → chips in the Labels field)
    "vlm-annotated":        "",
    "head-only-finetune":   "",

    # Properties (non-empty)
    "registered_at":  datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),

    # Training hyperparameters
    "param.epochs":    _fmt(train_args.get("epochs"), digits=0),
    "param.imgsz":     _fmt(train_args.get("imgsz"), digits=0),
    "param.batch":     _fmt(train_args.get("batch"), digits=0),
    "param.lr0":       _fmt(train_args.get("lr0"), digits=6),
    "param.optimizer": str(train_args.get("optimizer", "")),
    "param.freeze":    _fmt(train_args.get("freeze"), digits=0),
    "param.patience":  _fmt(train_args.get("patience"), digits=0),

    # Final-epoch metrics (whole-dataset, all classes)
    "metric.final_epoch": _fmt(final_row.get("epoch"), digits=0),
    "metric.mAP50":       _fmt(final_row.get("metrics/mAP50(B)")),
    "metric.mAP50_95":    _fmt(final_row.get("metrics/mAP50-95(B)")),
    "metric.precision":   _fmt(final_row.get("metrics/precision(B)")),
    "metric.recall":      _fmt(final_row.get("metrics/recall(B)")),
}

# --- Connect to the RHOAI Model Registry ---
# `server_address` is just scheme+host — do NOT include port or API path; the
# client appends them itself.
MODEL_REGISTRY_URL = os.environ.get(
    "MODEL_REGISTRY_URL",
    "https://user1-registry-rest.apps.ocp.9xgvv.sandbox3434.opentlc.com",
)
registry = ModelRegistry(
    server_address=MODEL_REGISTRY_URL,
    port=int(os.environ.get("MODEL_REGISTRY_PORT", 443)),
    author=os.environ.get("JUPYTER_USER", "workshop"),
    user_token=os.environ.get("MODEL_REGISTRY_TOKEN"),
    is_secure=True,
)

version_description = (
    "YOLOv8n fine-tuned on Qwen2.5-VL zero-shot annotations, exported to ONNX. "
    "7 PPE-compliance classes. "
    f"mAP50={metadata['metric.mAP50'] or '?'} "
    f"after {metadata['param.epochs'] or '?'} epochs "
    f"(fine-tuning head only, backbone frozen)."
)

# `register_model` creates three linked objects and attaches the inputs as:
#   description / metadata       → ModelVersion (custom_properties)
#   uri / storage_key / storage_path / model_format_* → ModelArtifact
#   name                         → RegisteredModel (the "folder"; left blank otherwise)
#
# IMPORTANT: pass `storage_key` (name of the Data Connection Secret) and
# `storage_path` (S3 prefix) so the RHOAI dashboard renders the structured
# "Model location" panel (Endpoint / Region / Bucket / Path) instead of just a
# raw URI. The endpoint / region / bucket are read from the named Secret.
DATA_CONNECTION = os.environ.get("DATA_CONNECTION", SERVICE_ACCOUNT if "SERVICE_ACCOUNT" in dir() else "aws-connection-storage")
rm = registry.register_model(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    model_format_name="onnx",       # matches the OVMS supportedModelFormats entry
    model_format_version="1",
    storage_key=DATA_CONNECTION,    # Data Connection Secret → Endpoint/Region/Bucket
    storage_path=S3_PREFIX,         # "models/ppe-yolov8n-vlm" → rendered as Path
    model_source_kind="s3",
    model_source_class="ultralytics",
    model_source_group="yolov8",
    model_source_id=MODEL_NAME,
    model_source_name="model.onnx",
    uri=s3_uri,                     # keeps the raw URI queryable via the API
    description=version_description,
    metadata=metadata,
)

# --- Populate the parent RegisteredModel (Overview tab in the dashboard) ---
# Same label/property convention as above.
rm.description = (
    "PPE-compliance object detector (YOLOv8n, ONNX export). "
    "7 classes covering person + helmet/vest/gloves presence & absence. "
    "Trained on Qwen2.5-VL zero-shot annotations over a 40-image "
    "construction-PPE subset. Workshop demo — not production-hardened."
)
rm.owner = os.environ.get("JUPYTER_USER", "workshop")
rm.custom_properties = {
    # Labels (empty string values)
    "computer-vision":   "",
    "object-detection":  "",
    "ppe-compliance":    "",
    "vlm-annotated":     "",
    "workshop-demo":     "",
    # Properties (non-empty)
    "framework":      "ultralytics",
    "architecture":   "yolov8n",
    "export_format":  "onnx (opset 13)",
    "base_model":     "yolov8n.pt (COCO-pretrained)",
    "annotator":      "Qwen2.5-VL-7B-Instruct",
    "num_classes":    "7",
    "classes":        "person,helmet,no-helmet,vest,no-vest,gloves,no-gloves",
    "source_dataset": "construction-ppe (VLM-annotated subset)",
}
registry.update(rm)

# --- Audit printout: what the workshop audience will see in the RHOAI dashboard ---
print("Registered:", rm.name, "(id=", rm.id, ")")
print(f"Model location → storage_key={DATA_CONNECTION!r}, storage_path={S3_PREFIX!r}")
print("\nRegisteredModel (Overview tab):")
print(f"  description: {rm.description}")
print(f"  owner:       {rm.owner}")
for k, v in sorted(rm.custom_properties.items()):
    kind = "label" if v == "" else "prop "
    print(f"  {kind} {k:18s} = {v!r}")
print("\nModelVersion custom properties (Version 1 tab):")
for k in sorted(metadata):
    kind = "label" if metadata[k] == "" else "prop "
    print(f"  {kind} {k:24s} = {metadata[k]!r}")

## 5c. Deploy with KServe + OpenVINO Model Server

Two Kubernetes objects, created in order:

1. **`ServingRuntime`** (name: `ovms`) — OpenVINO Model Server. Only created if it doesn't already exist in this project. Serves `onnx`, `openvino_ir`, `tensorflow`, `paddle`, and `pytorch` formats over KServe v2 / gRPC-v2.
2. **`InferenceService`** — points the runtime at our S3 `storageUri` and exposes a route.

KServe pulls the model using S3 credentials from the `aws-connection-storage` Secret, referenced via the `serviceAccountName` of the same name (created by the RHOAI Data Connection).

In [ ]:
NAMESPACE        = os.environ.get("KUBERNETES_NAMESPACE", "ppe-detection-cv")
SERVING_RUNTIME  = os.environ.get("SERVING_RUNTIME", "ovms")
SERVICE_ACCOUNT  = os.environ.get("SERVICE_ACCOUNT", "aws-connection-storage")
OVMS_IMAGE       = os.environ.get(
    "OVMS_IMAGE",
    "registry.redhat.io/rhoai/odh-openvino-model-server-rhel9:v2025.4",
)

# --- (1) ServingRuntime: OpenVINO Model Server ---
# Mirrors the RHOAI `kserve-ovms` template (same spec the Fraud demo uses).
runtime_manifest = {
    "apiVersion": "serving.kserve.io/v1alpha1",
    "kind": "ServingRuntime",
    "metadata": {
        "name": SERVING_RUNTIME,
        "namespace": NAMESPACE,
        "labels": {"opendatahub.io/dashboard": "true"},
        "annotations": {
            "opendatahub.io/apiProtocol": "REST",
            "opendatahub.io/template-name": "kserve-ovms",
            "opendatahub.io/template-display-name": "OpenVINO Model Server",
            "opendatahub.io/serving-runtime-scope": "project",
        },
    },
    "spec": {
        "annotations": {
            "opendatahub.io/kserve-runtime": "ovms",
            "prometheus.io/path": "/metrics",
            "prometheus.io/port": "8888",
        },
        "containers": [{
            "name": "kserve-container",
            "image": OVMS_IMAGE,
            "args": [
                "--model_name={{.Name}}",
                "--port=8001",
                "--rest_port=8888",
                "--model_path=/mnt/models",
                "--file_system_poll_wait_seconds=0",
                "--metrics_enable",
            ],
            "ports": [{"containerPort": 8888, "protocol": "TCP"}],
        }],
        "multiModel": False,
        "protocolVersions": ["v2", "grpc-v2"],
        "supportedModelFormats": [
            {"name": "onnx",         "version": "1",       "autoSelect": True},
            {"name": "openvino_ir",  "version": "opset13", "autoSelect": True},
            {"name": "tensorflow",   "version": "2",       "autoSelect": True},
            {"name": "paddle",       "version": "2",       "autoSelect": True},
            {"name": "pytorch",      "version": "2",       "autoSelect": True},
        ],
    },
}

# --- (2) InferenceService: targets the runtime above ---
isvc_manifest = {
    "apiVersion": "serving.kserve.io/v1beta1",
    "kind": "InferenceService",
    "metadata": {
        "name": MODEL_NAME,
        "namespace": NAMESPACE,
        "labels": {"opendatahub.io/dashboard": "true"},
        "annotations": {
            "serving.knative.openshift.io/enablePassthrough": "true",
            "sidecar.istio.io/inject": "true",
            "sidecar.istio.io/rewriteAppHTTPProbers": "true",
        },
    },
    "spec": {
        "predictor": {
            "serviceAccountName": SERVICE_ACCOUNT,
            "model": {
                "runtime":     SERVING_RUNTIME,
                "modelFormat": {"name": "onnx", "version": "1"},
                "storageUri":  s3_uri,
                "resources": {
                    "requests": {"cpu": "500m", "memory": "1Gi"},
                    "limits":   {"cpu": "2",   "memory": "4Gi"},
                },
            },
        }
    },
}

print("ServingRuntime:", runtime_manifest["metadata"]["name"])
print("InferenceService:", isvc_manifest["metadata"]["name"])
print("storageUri:", s3_uri)

In [ ]:
from kubernetes import client, config

try:
    config.load_incluster_config()
except config.ConfigException:
    config.load_kube_config()

api = client.CustomObjectsApi()

def apply(group, version, plural, body, *, name):
    try:
        api.create_namespaced_custom_object(group, version, NAMESPACE, plural, body)
        print(f"Created {plural[:-1]} {name!r}")
    except client.ApiException as e:
        if e.status == 409:
            api.patch_namespaced_custom_object(group, version, NAMESPACE, plural, name, body)
            print(f"Patched existing {plural[:-1]} {name!r}")
        else:
            raise

# (1) ServingRuntime (v1alpha1)
apply("serving.kserve.io", "v1alpha1", "servingruntimes",
      runtime_manifest, name=SERVING_RUNTIME)

# (2) InferenceService (v1beta1)
apply("serving.kserve.io", "v1beta1", "inferenceservices",
      isvc_manifest, name=MODEL_NAME)

# Track the ISVC group/version/plural for the wait-loop below.
group, version, plural = "serving.kserve.io", "v1beta1", "inferenceservices"

## Wait until the InferenceService is Ready, then print the route

In [ ]:
TIMEOUT_S = 600
deadline  = time.time() + TIMEOUT_S
url = None

while time.time() < deadline:
    isvc = api.get_namespaced_custom_object(group, version, NAMESPACE, plural, MODEL_NAME)
    status = isvc.get("status", {})
    ready = next(
        (c for c in status.get("conditions", []) if c.get("type") == "Ready"),
        None,
    )
    if ready and ready.get("status") == "True":
        url = status.get("url") or status.get("address", {}).get("url")
        break
    print(f"  … waiting, Ready={ready.get('status') if ready else 'unknown'}")
    time.sleep(10)

if not url:
    raise TimeoutError(f"InferenceService {MODEL_NAME} did not reach Ready=True in {TIMEOUT_S}s.")

print("\nEndpoint:", url)

# Persist for the next notebook.
Path("output").mkdir(exist_ok=True)
Path("output/inference_endpoint.txt").write_text(url)

---

**Next:** [06-consume-deployed-endpoint.ipynb](06-consume-deployed-endpoint.ipynb) — POST a tensor at this endpoint (OVMS speaks the KServe v2 `/v2/models/<name>/infer` protocol with NCHW float32 input) and draw the predicted boxes.

> Note: the next notebook needs updating for the OVMS tensor-in/tensor-out wire format (YOLO preprocessing + postprocessing move client-side). Not yet done — that is task (b) from the plan.